In [ ]:
from virtual_accelerator.models.cu_hxr import get_cu_hxr_staged_model
import matplotlib.pyplot as plt
import numpy as np

import logging

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("pytao").setLevel(logging.WARNING)

model = get_cu_hxr_staged_model(end_element="OTR4", n_particles=10000)

# do a quadrupole scan using QUAD:IN20:525:BCTRL and plot the beam distribution at OTR4
OTR_IMAGE_PV = "OTRS:IN20:571:Image:ArrayData"
SCAN_QUAD_PV = "QUAD:IN20:525:BCTRL"

quad_values = np.linspace(-10, 10, 5)
images = []
for i, quad_value in enumerate(quad_values):
    logging.debug(f"Setting {SCAN_QUAD_PV} to {quad_value:.1f}")
    model.set({SCAN_QUAD_PV: quad_value})
    image = model.get(OTR_IMAGE_PV)
    images.append(image)

images = np.array(images)
max_value = np.max(images)

fig, ax = plt.subplots(1, len(quad_values), sharex=True, sharey=True)

# plot the images - zoomed in on the center 300x300 pixels
for i, (quad_value, image) in enumerate(zip(quad_values, images)):
    width = 300
    image = image[
        images.shape[1] // 2 - width // 2 : images.shape[1] // 2 + width // 2,
        images.shape[2] // 2 - width // 2 : images.shape[2] // 2 + width // 2,
    ]

    ax[i].imshow(
        image,
        rasterized=True,
        origin="lower",
    )
    ax[i].set_title(f"{quad_value:.1f}")
    ax[i].set_xlabel("X (pixels)")
    if i == 0:
        ax[i].set_ylabel("Y (pixels)")

# add label for quad strengths
ax[0].text(
    -0.1,
    1.05,
    "BCTRL (kG)",
    transform=ax[0].transAxes,
    ha="center",
    va="bottom",
)

plt.tight_layout()
fig.set_size_inches(10, 3)
fig.savefig("quad_scan.png", dpi=300)
fig.savefig("quad_scan.svg", dpi=300)